# YOLOv8-cls Manual Quantization Tester

This notebook verifies the manual quantization pipeline of the YOLOv8 classification model using a RAM-safe subsampled subset of the validation dataset in `dataset/val/` to prevent Out-Of-Memory (OOM) errors.

In [15]:
import os
import sys
import numpy as np
import onnx
import onnxruntime as ort
import tensorflow as tf
from tensorflow import keras
from PIL import Image
from ultralytics import YOLO

# Ensure project root is in python search path
sys.path.append(os.getcwd())

from onnx2keras.onnx2keras import onnx_to_keras
from quantization.quant_container import QuantContainer

In [16]:
def sanitize_onnx_model(model):
    """
    Sanitizes ONNX model tensor names by replacing '/' and ':' with '_' to comply with Keras 3.0.
    """
    def rename(name):
        return name.replace('/', '_').replace(':', '_')
        
    for inp in model.graph.input:
        inp.name = rename(inp.name)
    for out in model.graph.output:
        out.name = rename(out.name)
    for init in model.graph.initializer:
        init.name = rename(init.name)
    for node in model.graph.node:
        if node.name:
            node.name = rename(node.name)
        inputs = [rename(i) for i in node.input]
        del node.input[:]
        node.input.extend(inputs)
        
        outputs = [rename(o) for o in node.output]
        del node.output[:]
        node.output.extend(outputs)
    for vi in model.graph.value_info:
        vi.name = rename(vi.name)
    return model

In [17]:
def load_subsampled_dataset(dataset_dir, limit_per_class=2, limit_classes=10, batch_size=1):
    """
    Loads a subsampled slice of ImageNet to prevent OOM in RAM.
    - limit_per_class: Max images to load per folder class
    - limit_classes: Max classes (folders) to scan (None to load all 1000)
    """
    if not os.path.exists(dataset_dir):
        print(f"Dataset directory '{dataset_dir}' not found. Using dummy inputs.")
        for _ in range(3):
            yield np.random.uniform(0.0, 1.0, (batch_size, 3, 224, 224)).astype(np.float32), None
        return
        
    image_paths = []
    labels = []
    class_names = sorted([d for d in os.listdir(dataset_dir) if os.path.isdir(os.path.join(dataset_dir, d))])
    
    if limit_classes is not None:
        class_names = class_names[:limit_classes]
        
    class_to_idx = {name: i for i, name in enumerate(class_names)}
    
    for class_name in class_names:
        class_path = os.path.join(dataset_dir, class_name)
        count = 0
        for fname in os.listdir(class_path):
            if fname.lower().endswith(('.png', '.jpg', '.jpeg', '.bmp')):
                image_paths.append(os.path.join(class_path, fname))
                labels.append(class_to_idx[class_name])
                count += 1
                if count >= limit_per_class:
                    break
                    
    if not image_paths:
        print(f"No valid images found in '{dataset_dir}'. Using dummy inputs.")
        for _ in range(3):
            yield np.random.uniform(0.0, 1.0, (batch_size, 3, 224, 224)).astype(np.float32), None
        return
        
    print(f"Subsampled dataset loaded: {len(image_paths)} images across {len(class_names)} classes.")
    
    for i in range(0, len(image_paths), batch_size):
        batch_paths = image_paths[i:i+batch_size]
        batch_labels = labels[i:i+batch_size]
        
        batch_imgs = []
        for path in batch_paths:
            try:
                img = Image.open(path).convert('RGB')
                img = img.resize((224, 224))
                img_arr = np.array(img).astype(np.float32) / 255.0
                img_arr = np.transpose(img_arr, (2, 0, 1))
                batch_imgs.append(img_arr)
            except Exception as e:
                print(f"Failed to load image {path}: {e}")
                
        if batch_imgs:
            yield np.array(batch_imgs), np.array(batch_labels)

## Step 1 & 2: Load Model and Export to ONNX

In [18]:
print("Loading YOLOv8 classification model from Ultralytics...")
model = YOLO('yolov8n-cls.pt')

print("Exporting model to ONNX format...")
onnx_path = model.export(format='onnx', imgsz=224)
print(f"ONNX model saved at: {onnx_path}")

Loading YOLOv8 classification model from Ultralytics...
Exporting model to ONNX format...
Ultralytics 8.4.87  Python-3.13.1 torch-2.12.1+cpu CPU (Intel Core i5-10300H 2.50GHz)
YOLOv8n-cls summary (fused): 30 layers, 2,715,880 parameters, 0 gradients, 4.3 GFLOPs

PyTorch: starting from 'yolov8n-cls.pt' with input shape (1, 3, 224, 224) BCHW and output shape(s) (1, 1000) (5.3 MB)

ONNX: starting export with onnx 1.22.0 opset 20...
ONNX: slimming with onnxslim 0.1.94...
ONNX: export success  0.4s, saved as 'yolov8n-cls.onnx' (10.4 MB)

Export complete (0.5s)
Results saved to E:\phenikaa\quant_anything\yolov8n-cls.onnx
Predict:         yolo predict task=classify model=yolov8n-cls.onnx imgsz=224 
Validate:        yolo val task=classify model=yolov8n-cls.onnx imgsz=224 data=../datasets/imagenet  
Visualize:       https://netron.app
ONNX model saved at: yolov8n-cls.onnx


## Step 3 & 4: Convert to Keras 3.0

In [19]:
print("Loading ONNX model and sanitizing names...")
onnx_model = onnx.load(onnx_path)
sanitized_onnx = sanitize_onnx_model(onnx_model)

print("Converting ONNX to Keras 3.0 model...")
keras_model = onnx_to_keras(
    sanitized_onnx,
    input_names=['images'],
    input_shapes=[[3, 224, 224]],
    verbose=False,
    change_ordering=False
)
print("Model converted successfully!")

Trying to convert multi-output node


Loading ONNX model and sanitizing names...
Converting ONNX to Keras 3.0 model...


Trying to convert multi-output node
Trying to convert multi-output node
Trying to convert multi-output node


Model converted successfully!


## Step 5 & 6: Calibrate Quantization Container

In [20]:
print("Initializing QuantContainer...")
quant_graph = QuantContainer(keras_model, quant_type="int8")

# Create small set of calibration inputs
print("Generating calibration inputs...")
calib_data = [np.random.uniform(0.0, 1.0, (4, 3, 224, 224)).astype(np.float32) for _ in range(3)]

print("Calibrating graph...")
quant_graph.calibrate(calib_data)
print("Calibration completed.")

Initializing QuantContainer...
Generating calibration inputs...
Calibrating graph...
Starting calibration forward passes...
Finalizing calibration scale and zero-point parameters...
Calibration completed successfully.
Calibration completed.


## Step 7: Run Model Evaluation (ONNX vs Keras vs Quantized)

In [21]:
print("Initializing ONNX Runtime session...")
ort_session = ort.InferenceSession(onnx_path)

# Evaluate on a subset of the dataset: 2 images per class for 5 classes (total 10 images)
# This checks execution across different labels without loading all 1000 folders into memory
val_loader = load_subsampled_dataset(
    dataset_dir="dataset/val",
    limit_per_class=2,
    limit_classes=5,
    batch_size=1
)

onnx_keras_errors = []
keras_quant_errors = []
total_samples = 0
has_labels = False

for batch_idx, (x_batch, y_batch) in enumerate(val_loader):
    if y_batch is not None:
        has_labels = True
        
    # ONNX
    ort_inputs = {ort_session.get_inputs()[0].name: x_batch}
    ort_outputs = ort_session.run(None, ort_inputs)
    out_onnx = ort_outputs[0]
    
    # Keras
    out_keras = keras_model(tf.convert_to_tensor(x_batch)).numpy()
    
    # Quantized
    out_quant = quant_graph.quantized_infer(x_batch)
    
    # Max absolute differences
    diff_ok = np.max(np.abs(out_onnx - out_keras))
    diff_kq = np.max(np.abs(out_keras - out_quant))
    
    onnx_keras_errors.append(diff_ok)
    keras_quant_errors.append(diff_kq)
    total_samples += x_batch.shape[0]
    
    print(f"Sample {batch_idx + 1}: ONNX-Keras Diff = {diff_ok:.6f} | Keras-Quant Diff = {diff_kq:.6f}")

print("\n--- Final Metrics ---")
print(f"Total validation samples evaluated: {total_samples}")
print(f"Average ONNX vs Keras Max Difference: {np.mean(onnx_keras_errors):.6f}")
print(f"Average Keras vs Quantized Max Difference: {np.mean(keras_quant_errors):.6f}")

Initializing ONNX Runtime session...
Subsampled dataset loaded: 10 images across 5 classes.
Sample 1: ONNX-Keras Diff = 0.000001 | Keras-Quant Diff = 0.542347
Sample 2: ONNX-Keras Diff = 0.000001 | Keras-Quant Diff = 0.748653
Sample 3: ONNX-Keras Diff = 0.000000 | Keras-Quant Diff = 0.965523
Sample 4: ONNX-Keras Diff = 0.000000 | Keras-Quant Diff = 0.118228
Sample 5: ONNX-Keras Diff = 0.000000 | Keras-Quant Diff = 0.801274
Sample 6: ONNX-Keras Diff = 0.000001 | Keras-Quant Diff = 0.743158
Sample 7: ONNX-Keras Diff = 0.000001 | Keras-Quant Diff = 0.420132
Sample 8: ONNX-Keras Diff = 0.000001 | Keras-Quant Diff = 0.294232
Sample 9: ONNX-Keras Diff = 0.000001 | Keras-Quant Diff = 0.292337
Sample 10: ONNX-Keras Diff = 0.000000 | Keras-Quant Diff = 0.928829

--- Final Metrics ---
Total validation samples evaluated: 10
Average ONNX vs Keras Max Difference: 0.000001
Average Keras vs Quantized Max Difference: 0.585471
